# EN4554 Assignment 1, Part 1: LoRA+ on SmolVLM-256M
## Notebook 05: every number in the report, recomputed from the saved results

This notebook trains nothing and needs **no GPU** (about 1 minute on a CPU). It reads the result files written by notebooks 01–04 and recomputes every accuracy, step count and confidence interval quoted in the report.

**Statistics.** All comparisons are **paired**: both settings answer the same 500 evaluation questions, so we bootstrap over questions (20,000 resamples) and report a 95% confidence interval for the *difference*. Fine-tuned settings use each question's score averaged over the 3 fresh seeds. Comparisons *between tasks* (a difference of differences) resample each task's questions independently. Every interval uses its own fixed random seed, so each number is reproducible on its own.

Files needed in `RES_DIR`: `zeroshot.json`, `preds_zeroshot_*.jsonl` (notebook 01), `grid_*.jsonl` (02), `seeds_*.jsonl` (03), `resolution_*.jsonl` (04).

In [1]:
import os, sys, re, json
import numpy as np

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RES_DIR = "/content/drive/MyDrive/EN4554_LoRAplus/results"
else:   # local copy of the Drive results folder
    RES_DIR = next(p for p in ("../results/results", "../results", "results", "EN4554_LoRAplus/results")
                   if os.path.exists(f"{p}/seeds_summary.json"))
print("reading results from", RES_DIR)
TASKS = ("textvqa", "vqav2")

# ---- VQA accuracy (identical to notebooks 01-04) ----
_CONTRACTIONS = {"dont": "don't", "doesnt": "doesn't", "isnt": "isn't", "arent": "aren't", "cant": "can't",
                 "wont": "won't", "didnt": "didn't", "wasnt": "wasn't", "werent": "weren't", "couldnt": "couldn't",
                 "shouldnt": "shouldn't", "wouldnt": "wouldn't", "hasnt": "hasn't", "havent": "haven't",
                 "im": "i'm", "ive": "i've", "youre": "you're", "theyre": "they're", "thats": "that's",
                 "whats": "what's", "wheres": "where's", "hes": "he's", "shes": "she's", "lets": "let's"}
_NUMS = {"none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5",
         "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10"}
_ARTICLES = {"a", "an", "the"}
_PUNCT = [";", "/", "[", "]", '"', "{", "}", "(", ")", "=", "+", "\\", "_", "-", ">", "<", "@", "`", ",", "?", "!"]
_PERIOD = re.compile(r"(?!<=\d)(\.)(?!\d)")
_COMMA = re.compile(r"(\d)(,)(\d)")

def normalize_answer(s):
    s = s.replace("\n", " ").replace("\t", " ").strip().lower()
    out = s
    for p in _PUNCT:
        if (p + " " in s or " " + p in s) or _COMMA.search(s):
            out = out.replace(p, "")
        else:
            out = out.replace(p, " ")
    out = _PERIOD.sub("", out)
    words = []
    for w in out.split():
        w = _NUMS.get(w, w)
        if w in _ARTICLES:
            continue
        words.append(_CONTRACTIONS.get(w, w))
    return " ".join(words)

def vqa_score(pred, answers):
    p = normalize_answer(pred)
    gts = [normalize_answer(a) for a in answers]
    return float(np.mean([min(1.0, sum(g == p for g in gts[:i] + gts[i + 1:]) / 3.0) for i in range(len(gts))]))

# ---- paired bootstrap ----
N_BOOT = 20000

def ci(d, seed=0):
    # mean of per-question differences d, with a 95% bootstrap CI (in accuracy points)
    m = d[np.random.default_rng(seed).integers(0, len(d), size=(N_BOOT, len(d)))].mean(1)
    return 100 * d.mean(), 100 * np.percentile(m, 2.5), 100 * np.percentile(m, 97.5)

def ci_between_tasks(d1, d2, seed=0):
    # mean(d1) - mean(d2) for two different question sets: each task is resampled independently
    rng = np.random.default_rng(seed)
    m1 = d1[rng.integers(0, len(d1), size=(N_BOOT, len(d1)))].mean(1)
    m2 = d2[rng.integers(0, len(d2), size=(N_BOOT, len(d2)))].mean(1)
    return 100 * (d1.mean() - d2.mean()), 100 * np.percentile(m1 - m2, 2.5), 100 * np.percentile(m1 - m2, 97.5)

def fmt(t):
    return f"{t[0]:+.1f} [{t[1]:+.1f}, {t[2]:+.1f}]"

def jl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

reading results from results


In [2]:
# ---- load everything and re-score every prediction with the same metric ----
ANS, ZS, FT, SEEDREC, RES, RESINFO, GRID = {}, {}, {}, {}, {}, {}, {}
for t in TASKS:
    z = jl(f"{RES_DIR}/preds_zeroshot_{t}.jsonl")
    ANS[t] = [r["answers"] for r in z]
    ZS[t] = np.array([vqa_score(r["pred"], r["answers"]) for r in z])
    assert np.allclose(ZS[t], [r["score"] for r in z]), "zero-shot re-scoring differs from notebook 01"
    SEEDREC[t] = jl(f"{RES_DIR}/seeds_{t}.jsonl")
    for name in dict.fromkeys(r["name"] for r in SEEDREC[t]):
        rs = sorted((r for r in SEEDREC[t] if r["name"] == name), key=lambda r: r["seed"])
        FT[t, name] = np.array([[vqa_score(p, a) for p, a in zip(r["preds"], ANS[t])] for r in rs])   # seeds x questions
        assert np.allclose(FT[t, name].mean(1) * 100, [r["acc"] for r in rs], atol=0.01), name
    GRID[t] = jl(f"{RES_DIR}/grid_{t}.jsonl")
    if os.path.exists(f"{RES_DIR}/resolution_{t}.jsonl"):
        for r in jl(f"{RES_DIR}/resolution_{t}.jsonl"):
            RES[t, r["setting"]] = np.array([vqa_score(p, a) for p, a in zip(r["preds"], ANS[t])])
            RESINFO[t, r["setting"]] = r
            assert abs(RES[t, r["setting"]].mean() * 100 - r["acc"]) < 0.01, (t, r["setting"])
    print(f"{t:8s}: zero-shot {100*ZS[t].mean():.2f} | {len(GRID[t])} grid runs | "
          f"{len(SEEDREC[t])} seed runs | {sum(k[0] == t for k in RES)} resolution settings")
print("all saved accuracies reproduced exactly by re-scoring the saved predictions")

def ft(t, name):
    return FT[t, name].mean(0)          # per-question score averaged over the 3 seeds

textvqa : zero-shot 32.76 | 30 grid runs | 12 seed runs | 5 resolution settings


vqav2   : zero-shot 45.26 | 15 grid runs | 9 seed runs | 5 resolution settings
all saved accuracies reproduced exactly by re-scoring the saved predictions


## 1. Fresh-seed results, H1 and H2 (report Table 1)
*Steps* = optimiser steps until the 10-step moving average of the training loss falls below 1.0.

In [3]:
def steps_below(losses, thr=1.0, k=10):
    s = np.convolve(np.asarray(losses, float), np.ones(k) / k, mode="valid")
    return int(np.argmax(s < thr)) if (s < thr).any() else None

for t in TASKS:
    grid = {(r["eta_a"], r["ratio"]): r["acc"] for r in GRID[t]}
    print(f"\n{t}  (zero-shot {100*ZS[t].mean():.1f})")
    print(f"  {'setting':12s} {'eta_A':>6s} {'lam':>3s} | {'acc mean ± sd':>14s} | steps (per seed)  | seed-42 grid -> fresh seeds")
    for name in dict.fromkeys(r["name"] for r in SEEDREC[t]):
        rs = sorted((r for r in SEEDREC[t] if r["name"] == name), key=lambda r: r["seed"])
        accs = [r["acc"] for r in rs]
        sp = [steps_below(r["losses"]) for r in rs]
        print(f"  {name:12s} {rs[0]['eta_a']:6.0e} {rs[0]['ratio']:3g} | {np.mean(accs):6.1f} ± {np.std(accs, ddof=1):.1f}   "
              f"| {np.mean(sp):3.0f} {str(sp):12s} | {grid[(rs[0]['eta_a'], rs[0]['ratio'])]:.1f} -> {np.mean(accs):.1f}")

print("\nBaseline vs adapted (tuned LoRA - zero-shot):")
for t in TASKS:
    print(f"  {t:8s} {fmt(ci(ft(t, 'LoRA-best') - ZS[t]))}")
print("H1 (tuned LoRA+ - tuned LoRA):")
for t in TASKS:
    d = ft(t, "LoRA+-best") - ft(t, "LoRA-best")
    per_seed = 100 * (FT[t, "LoRA+-best"].mean(1) - FT[t, "LoRA-best"].mean(1))
    print(f"  {t:8s} {fmt(ci(d))}   per seed {np.round(per_seed, 1).tolist()}")
print("Same eta_A = 1e-4 (LoRA+ lambda=16 - LoRA):")
print(f"  textvqa  {fmt(ci(ft('textvqa', 'LoRA+-lowlr') - ft('textvqa', 'LoRA-lowlr')))}")
print(f"  vqav2    {fmt(ci(ft('vqav2', 'LoRA+-best') - ft('vqav2', 'LoRA-lowlr')))}   (VQAv2 tuned LoRA+ is this setting)")
h2 = ci_between_tasks(ft("textvqa", "LoRA+-best") - ft("textvqa", "LoRA-best"), ft("vqav2", "LoRA+-best") - ft("vqav2", "LoRA-best"))
print(f"H2 (LoRA+ gain on TextVQA - LoRA+ gain on VQAv2): {fmt(h2)}   (H2 predicts > 0)")

print("\nAccuracy on 200 questions during training (mean of 3 seeds):")
for t in TASKS:
    for name in dict.fromkeys(r["name"] for r in SEEDREC[t]):
        rs = [r for r in SEEDREC[t] if r["name"] == name]
        steps = [s for s, _ in rs[0]["curve"]]
        print(f"  {t:8s} {name:12s}", dict(zip(steps, np.round(np.mean([[a for _, a in r['curve']] for r in rs], 0), 1).tolist())))


textvqa  (zero-shot 32.8)
  setting       eta_A lam |  acc mean ± sd | steps (per seed)  | seed-42 grid -> fresh seeds
  LoRA-best     8e-04   1 |   41.1 ± 0.7   |  20 [13, 25, 22] | 42.8 -> 41.1
  LoRA+-best    4e-04   8 |   39.4 ± 0.3   |  15 [13, 10, 22] | 42.9 -> 39.4
  LoRA-lowlr    1e-04   1 |   40.1 ± 0.5   |  35 [43, 31, 30] | 40.6 -> 40.1
  LoRA+-lowlr   1e-04  16 |   40.7 ± 0.8   |  16 [13, 13, 22] | 41.5 -> 40.7

vqav2  (zero-shot 45.3)
  setting       eta_A lam |  acc mean ± sd | steps (per seed)  | seed-42 grid -> fresh seeds
  LoRA-best     4e-04   1 |   63.4 ± 0.9   |  13 [13, 12, 14] | 64.9 -> 63.4
  LoRA+-best    1e-04  16 |   63.4 ± 0.3   |   7 [8, 6, 7]    | 65.1 -> 63.4
  LoRA-lowlr    1e-04   1 |   62.8 ± 0.8   |  37 [37, 29, 44] | 62.5 -> 62.8

Baseline vs adapted (tuned LoRA - zero-shot):
  textvqa  +8.3 [+4.7, +12.0]
  vqav2    +18.2 [+14.1, +22.2]
H1 (tuned LoRA+ - tuned LoRA):


  textvqa  -1.7 [-3.7, +0.3]   per seed [-2.1, -1.5, -1.4]
  vqav2    -0.0 [-1.7, +1.6]   per seed [-0.7, 0.5, 0.2]
Same eta_A = 1e-4 (LoRA+ lambda=16 - LoRA):
  textvqa  +0.6 [-1.5, +2.7]
  vqav2    +0.7 [-1.4, +2.7]   (VQAv2 tuned LoRA+ is this setting)


H2 (LoRA+ gain on TextVQA - LoRA+ gain on VQAv2): -1.7 [-4.3, +0.9]   (H2 predicts > 0)

Accuracy on 200 questions during training (mean of 3 seeds):
  textvqa  LoRA-best    {10: 25.2, 20: 37.9, 40: 36.9, 70: 39.5, 125: 40.2, 190: 40.2}
  textvqa  LoRA+-best   {10: 35.0, 20: 35.5, 40: 38.4, 70: 37.2, 125: 38.0, 190: 41.0}
  textvqa  LoRA-lowlr   {10: 30.6, 20: 29.0, 40: 36.1, 70: 37.9, 125: 39.0, 190: 39.6}
  textvqa  LoRA+-lowlr  {10: 29.0, 20: 37.3, 40: 37.6, 70: 39.8, 125: 42.3, 190: 41.1}
  vqav2    LoRA-best    {10: 41.9, 20: 59.6, 40: 64.3, 70: 64.5, 125: 66.7, 190: 66.5}
  vqav2    LoRA+-best   {10: 54.1, 20: 61.9, 40: 64.8, 70: 62.6, 125: 65.6, 190: 65.7}
  vqav2    LoRA-lowlr   {10: 40.9, 20: 41.0, 40: 60.0, 70: 62.0, 125: 63.6, 190: 65.9}


## 2. The seed-42 grid (report Figure 1)

In [4]:
for t in TASKS:
    ok = [r for r in GRID[t] if not r["diverged"]]
    best = max(ok, key=lambda r: r["acc"])
    best1 = max((r for r in ok if r["ratio"] == 1), key=lambda r: r["acc"])
    print(f"{t}: best overall {best['acc']:.1f} (eta_A={best['eta_a']:.0e}, lambda={best['ratio']:g}) | "
          f"best plain LoRA {best1['acc']:.1f} (eta_A={best1['eta_a']:.0e}) | difference {best['acc']-best1['acc']:+.1f}")
    rows = {}
    for r in ok:
        rows.setdefault(r["ratio"], []).append(r["acc"])
    print("   best accuracy per lambda row:", {k: round(max(v), 1) for k, v in sorted(rows.items())})
    low = sorted({r["eta_b"] for r in GRID[t] if r["diverged"] or r["acc"] < 30})
    print("   eta_B of collapsed/diverged runs (acc < 30):", [f"{x:.1e}" for x in low])
    nA = [r for r in ok if r["eta_a"] == 1e-4]
    print("   steps to loss < 1 at eta_A = 1e-4:", {r["ratio"]: steps_below(r["losses"]) for r in sorted(nA, key=lambda r: r["ratio"])})

textvqa: best overall 42.9 (eta_A=4e-04, lambda=8) | best plain LoRA 42.8 (eta_A=8e-04) | difference +0.1
   best accuracy per lambda row: {1: 42.8, 2: 42.3, 4: 42.9, 8: 42.9, 16: 41.5, 32: 38.4}
   eta_B of collapsed/diverged runs (acc < 30): ['1.3e-02', '2.6e-02', '5.1e-02']
   steps to loss < 1 at eta_A = 1e-4: {1: 35, 2: 35, 4: 19, 8: 14, 16: 13, 32: 13}
vqav2: best overall 65.1 (eta_A=1e-04, lambda=16) | best plain LoRA 64.9 (eta_A=4e-04) | difference +0.3
   best accuracy per lambda row: {1: 64.9, 4: 63.7, 16: 65.1}
   eta_B of collapsed/diverged runs (acc < 30): []
   steps to loss < 1 at eta_A = 1e-4: {1: 33, 4: 16, 16: 9}


## 3. Resolution test (notebook 04, report Table 2)
Zero-shot accuracy on the same 500 questions per task, re-loaded from the **full-resolution** originals, under five input settings. `tile_2048` is SmolVLM's own default (`do_image_splitting=True`, longest edge 2048). Our fine-tuning used the cheap `512_notiling` input (64 visual tokens).

* **P1** (pre-specified): at 256 px, TextVQA loses *more* than VQAv2. Test: change(TextVQA) − change(VQAv2) < 0.
* **P2** (pre-specified): with the default tiling, TextVQA gains *more* than VQAv2. Test: the same difference > 0.

In [5]:
SETTINGS = ["256_notiling", "512_notiling", "tile_1024", "tile_1536", "tile_2048"]
if RES:
    print(f"{'setting':13s} | {'tokens (T / V)':>15s} | {'TextVQA':>24s} | {'VQAv2':>24s} | TextVQA change - VQAv2 change")
    for k in SETTINGS:
        cells = []
        for t in TASKS:
            acc = 100 * RES[t, k].mean()
            cells.append(f"{acc:5.1f}" if k == "512_notiling" else f"{acc:5.1f} {fmt(ci(RES[t, k] - RES[t, '512_notiling']))}")
        dd = "" if k == "512_notiling" else fmt(ci_between_tasks(RES["textvqa", k] - RES["textvqa", "512_notiling"],
                                                                 RES["vqav2", k] - RES["vqav2", "512_notiling"]))
        tok = " / ".join(f"{RESINFO[t, k]['mean_image_tokens']:.0f}" for t in TASKS)
        print(f"{k:13s} | {tok:>15s} | {cells[0]:>24s} | {cells[1]:>24s} | {dd}")
    print("\nP1 (256 px):", fmt(ci_between_tasks(RES["textvqa", "256_notiling"] - RES["textvqa", "512_notiling"],
                                             RES["vqav2", "256_notiling"] - RES["vqav2", "512_notiling"])), " -> predicted < 0")
    print("P2 (tile_2048):", fmt(ci_between_tasks(RES["textvqa", "tile_2048"] - RES["textvqa", "512_notiling"],
                                              RES["vqav2", "tile_2048"] - RES["vqav2", "512_notiling"])), " -> predicted > 0")
    for t in TASKS:
        print(f"check {t}: 512_notiling from full-res originals {100*RES[t, '512_notiling'].mean():.1f} "
              f"vs notebook-01 zero-shot on the 512-px copies {100*ZS[t].mean():.1f}")

setting       |  tokens (T / V) |                  TextVQA |                    VQAv2 | TextVQA change - VQAv2 change


256_notiling  |         64 / 64 |   28.4 -3.3 [-6.7, +0.1] |   43.7 -1.6 [-4.7, +1.4] | -1.7 [-6.2, +3.0]
512_notiling  |         64 / 64 |                     31.6 |                     45.3 | 


tile_1024     |       317 / 318 |  48.8 +17.2 [+13.3, +21.2] |  64.3 +19.0 [+14.9, +23.2] | -1.8 [-7.5, +3.9]


tile_1536     |       570 / 570 |  50.6 +19.0 [+15.1, +22.9] |  65.0 +19.7 [+15.7, +23.7] | -0.7 [-6.3, +4.9]


tile_2048     |       882 / 853 |  49.1 +17.5 [+13.5, +21.5] |  64.4 +19.1 [+15.0, +23.2] | -1.6 [-7.3, +4.1]

P1 (256 px): -1.7 [-6.2, +3.0]  -> predicted < 0


P2 (tile_2048): -1.6 [-7.3, +4.1]  -> predicted > 0
check textvqa: 512_notiling from full-res originals 31.6 vs notebook-01 zero-shot on the 512-px copies 32.8
check vqav2: 512_notiling from full-res originals 45.3 vs notebook-01 zero-shot on the 512-px copies 45.3


## 4. Post-hoc: can fine-tuning stand in for image detail?
P2 asked whether detail helps TextVQA *more*. A different question is whether LoRA fine-tuning at 64 tokens recovers what the default tiling gives. If the tiling gain on a task is mostly *adaptation* (answer format, calibration, the input layout the model expects), an adapter can learn it. If it is *information* the 64 tokens do not contain, no adapter can. This analysis was designed **after** seeing the P1/P2 results, so it is exploratory.

1. **Accuracy:** zero-shot with tiling − LoRA fine-tuned at 512 (tuned LoRA, 3-seed mean), per task and between tasks.
2. **Share of the tiling gain recovered by fine-tuning:** (tuned LoRA − zero-shot) / (tiled − 512 px zero-shot).
3. **Do they fix the same questions?** Among questions the 512-px zero-shot model gets wrong (score 0), "fixed" means score ≥ 0.5 (for fine-tuning: the 3-seed mean). φ is the correlation between "fixed by tiling" and "fixed by fine-tuning". It corrects for chance overlap, which a raw overlap share does not: fine-tuning fixes more VQAv2 questions overall.

In [6]:
def phi(base, a, b, thr=0.5):
    w = base == 0
    return np.corrcoef(a[w] >= thr, b[w] >= thr)[0, 1]

def phi_ci(t1, t2, k, thr=0.5, n=4000, seed=0):
    # phi(t2) - phi(t1), each task resampled independently
    rng = np.random.default_rng(seed)
    def f(t, idx):
        return phi(RES[t, "512_notiling"][idx], RES[t, k][idx], ft(t, "LoRA-best")[idx], thr)
    full = {t: np.arange(len(ANS[t])) for t in (t1, t2)}
    bs = [f(t2, rng.integers(0, len(ANS[t2]), len(ANS[t2]))) - f(t1, rng.integers(0, len(ANS[t1]), len(ANS[t1]))) for _ in range(n)]
    return f(t1, full[t1]), f(t2, full[t2]), f(t2, full[t2]) - f(t1, full[t1]), *np.percentile(bs, [2.5, 97.5])

if RES:
    for k in ("tile_2048", "tile_1024", "tile_1536"):
        print(f"--- {k}" + ("  (SmolVLM default; headline)" if k == "tile_2048" else "  (robustness)"))
        for t in TASKS:
            print(f"  {t:8s} zero-shot tiled - fine-tuned LoRA @512: {fmt(ci(RES[t, k] - ft(t, 'LoRA-best')))}"
                  f"   | fine-tuning recovers {100 * (ft(t, 'LoRA-best') - ZS[t]).mean() / (RES[t, k] - RES[t, '512_notiling']).mean():.0f}% of the tiling gain")
        print("  between tasks (TextVQA - VQAv2):", fmt(ci_between_tasks(RES["textvqa", k] - ft("textvqa", "LoRA-best"),
                                                                     RES["vqav2", k] - ft("vqav2", "LoRA-best"))))
        for thr in (0.3, 0.5, 1.0):
            pT, pV, d, lo, hi = phi_ci("textvqa", "vqav2", k, thr)
            print(f"  phi (fixed-by-tiling vs fixed-by-fine-tuning, thr {thr}): TextVQA {pT:.2f}, VQAv2 {pV:.2f}, "
                  f"difference {d:+.2f} [{lo:+.2f}, {hi:+.2f}]")
        for t in TASKS:
            w = RES[t, "512_notiling"] == 0
            a, b = w & (RES[t, k] >= 0.5), w & (ft(t, "LoRA-best") >= 0.5)
            print(f"  {t:8s} wrong at 512: {w.sum()} | fixed by tiling only {(a & ~b).sum()}, by fine-tuning only {(b & ~a).sum()}, by both {(a & b).sum()}")

    # examples: TextVQA questions fixed by tiling but not by fine-tuning
    z = jl(f"{RES_DIR}/preds_zeroshot_textvqa.jsonl")
    k = "tile_2048"
    w = RES["textvqa", "512_notiling"] == 0
    idx = np.where(w & (RES["textvqa", k] >= 0.5) & (ft("textvqa", "LoRA-best") < 0.5))[0]
    s1 = sorted((r for r in SEEDREC["textvqa"] if r["name"] == "LoRA-best"), key=lambda r: r["seed"])[0]
    print(f"\nTextVQA questions fixed by tiling ({k}) but not by fine-tuning (first 12 of {len(idx)}):")
    print(f"  {'question':50s} | {'512 zero-shot':14s} | {'tiled':14s} | {'LoRA seed 1':14s} | most common answer")
    for i in idx[:12]:
        top = max(set(z[i]["answers"]), key=z[i]["answers"].count)
        print(f"  {z[i]['question'][:50]:50s} | {RESINFO['textvqa', '512_notiling']['preds'][i][:14]:14s} | "
              f"{RESINFO['textvqa', k]['preds'][i][:14]:14s} | {s1['preds'][i][:14]:14s} | {top}")

--- tile_2048  (SmolVLM default; headline)
  textvqa  zero-shot tiled - fine-tuned LoRA @512: +8.0 [+3.8, +12.3]   | fine-tuning recovers 48% of the tiling gain
  vqav2    zero-shot tiled - fine-tuned LoRA @512: +1.0 [-2.4, +4.4]   | fine-tuning recovers 95% of the tiling gain


  between tasks (TextVQA - VQAv2): +7.1 [+1.5, +12.5]


  phi (fixed-by-tiling vs fixed-by-fine-tuning, thr 0.3): TextVQA 0.23, VQAv2 0.57, difference +0.33 [+0.19, +0.48]


  phi (fixed-by-tiling vs fixed-by-fine-tuning, thr 0.5): TextVQA 0.20, VQAv2 0.51, difference +0.31 [+0.15, +0.47]


  phi (fixed-by-tiling vs fixed-by-fine-tuning, thr 1.0): TextVQA 0.21, VQAv2 0.49, difference +0.28 [+0.11, +0.44]
  textvqa  wrong at 512: 329 | fixed by tiling only 67, by fine-tuning only 41, by both 39
  vqav2    wrong at 512: 254 | fixed by tiling only 29, by fine-tuning only 33, by both 84
--- tile_1024  (robustness)
  textvqa  zero-shot tiled - fine-tuned LoRA @512: +7.7 [+3.7, +11.8]   | fine-tuning recovers 48% of the tiling gain
  vqav2    zero-shot tiled - fine-tuned LoRA @512: +0.9 [-2.5, +4.4]   | fine-tuning recovers 95% of the tiling gain


  between tasks (TextVQA - VQAv2): +6.8 [+1.5, +12.0]


... (3 outputs left out)

  between tasks (TextVQA - VQAv2): +8.0 [+2.6, +13.2]


  phi (fixed-by-tiling vs fixed-by-fine-tuning, thr 0.3): TextVQA 0.29, VQAv2 0.58, difference +0.29 [+0.15, +0.43]


  phi (fixed-by-tiling vs fixed-by-fine-tuning, thr 0.5): TextVQA 0.27, VQAv2 0.52, difference +0.25 [+0.09, +0.40]


  phi (fixed-by-tiling vs fixed-by-fine-tuning, thr 1.0): TextVQA 0.21, VQAv2 0.49, difference +0.28 [+0.12, +0.44]
  textvqa  wrong at 512: 329 | fixed by tiling only 66, by fine-tuning only 35, by both 45
  vqav2    wrong at 512: 254 | fixed by tiling only 29, by fine-tuning only 32, by both 85

TextVQA questions fixed by tiling (tile_2048) but not by fine-tuning (first 12 of 67):
  question                                           | 512 zero-shot  | tiled          | LoRA seed 1    | most common answer
  what number is on the red jersey?                  | 3.             | 6.             | 9              | 6
  what is steel-enforced on this garage door opener? | Chambered.     | Belt.          | chamberlain    | belt
  what color is the letter d on her jersey?          | Black.         | Blue.          | answering does | blue
  what word is on the front of the blue shirt?       | Man.           | Marine.        | mani           | marine
  what does the text say under the cup?       